This notebook verifies that final matched Bogard Lakes dataset doesn't have any mismatches with geocode info available

In [95]:
import geopandas as gpd
import pandas as pd
import numpy as np

from land_cover.load import loadBogardSuppl, bogard_output_path_raw

In [52]:
version = "v3"
_, out_dir, file_name = loadBogardSuppl()
out_spatial_stem = out_dir / "shp" / "qa_qc" / f"{file_name}_geocoded_{version}"
geocode_pth = f"{out_spatial_stem}.gpkg"
updated_gee_pth = "/Volumes/metis/ABOVE3/Tom/gee_input/updated/gee_cleaned_geocode_2025-07-10.csv"
print(geocode_pth)

/Volumes/metis/ABOVE3/Bogard_suppl_data/edk_out/shp/qa_qc/Bogard19_ESM_alldata_wh_geocoded_v3.gpkg


In [53]:
gdf_gee_input_pth = "/Volumes/metis/ABOVE3/Tom/gee_input/gee_cleaned_sample_data_2025-03-06.csv"

In [54]:
# Load gee table as DataFrame and convert to GeoDataFrame
gee_output_gdf = gpd.read_file(bogard_output_path_raw)

# Load geocode table as GeoDataFrame
geocode_gdf = gpd.read_file(geocode_pth)
len(gee_output_gdf)

# load bml shapefile for merging in index name from Tom
gdf_gee_input = pd.read_csv(gdf_gee_input_pth)

In [55]:
gdf_gee_input.head()
gdf_gee_input.info()
# gdf_gee_input.iloc[0,:]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8714 entries, 0 to 8713
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   system:index  8714 non-null   object 
 1   CurrentlyM    8714 non-null   int64  
 2   Latitude      8714 non-null   float64
 3   Longitude     8714 non-null   float64
 4   SampleUID     8714 non-null   object 
 5   Source        8714 non-null   object 
 6   WesternHem    8714 non-null   bool   
 7   .geo          8714 non-null   object 
dtypes: bool(1), float64(2), int64(1), object(4)
memory usage: 485.2+ KB


In [56]:
gee_output_gdf.head()

,PointLat,PointLon,caption,pld_match,sampleUID,savetype,geometry
0,48.128496885028184,-71.28594971031188,No Caption,-99999.31415,BML_00000060,New polygon,"POLYGON ((-71.28625 48.13021, -71.28599 48.130..."
1,48.48063681142981,-79.4085016785392,tiny beaver pond- approximate,-99999.31415,BML_00000097,New polygon,"POLYGON ((-79.40864 48.48067, -79.40856 48.480..."
2,-99999.31415,-99999.31415,"shield, bad google imagery",-99999.31415,BML_00000281,Mismatch,"POLYGON ((-0.00045 -0.00045, -0.00045 0.00045,..."
3,-99999.31415,-99999.31415,closest lake to this road point,8223660592,BML_00005825,MatchedPLD,"POLYGON ((-133.05027 68.08287, -133.05025 68.0..."
4,-99999.31415,-99999.31415,also digitized,7240067712,BML_00000060,MatchedPLD,"POLYGON ((-71.28654 48.12908, -71.28574 48.129..."


In [59]:
# join to bofard shape lakes in WH
gdf_gee_merged = gdf_gee_input.merge(
    geocode_gdf, right_on=["lat (decimal)", "long (decimal)"], left_on=["Latitude", "Longitude"]
).query("WesternHem == True and Source == 'BogardMapLakes' and CurrentlyM == 0")

In [60]:
gdf_gee_merged.info()

<class 'pandas.core.frame.DataFrame'>
Index: 101 entries, 12 to 112
Data columns (total 31 columns):
 #   Column                                      Non-Null Count  Dtype   
---  ------                                      --------------  -----   
 0   system:index                                101 non-null    object  
 1   CurrentlyM                                  101 non-null    int64   
 2   Latitude                                    101 non-null    float64 
 3   Longitude                                   101 non-null    float64 
 4   SampleUID                                   101 non-null    object  
 5   Source                                      101 non-null    object  
 6   WesternHem                                  101 non-null    bool    
 7   .geo                                        101 non-null    object  
 8   Category (1=same as YFB lakes/2=different)  63 non-null     float64 
 9   coscatsID                                   39 non-null     float64 
 10  Sbcode

FYI, all these geocoded lakes are in eastern NA, not in ABOVE region

In [71]:
geocoded = gdf_gee_merged.query("geocoder.notnull() ")
geocoded.info()

<class 'pandas.core.frame.DataFrame'>
Index: 15 entries, 12 to 100
Data columns (total 31 columns):
 #   Column                                      Non-Null Count  Dtype   
---  ------                                      --------------  -----   
 0   system:index                                15 non-null     object  
 1   CurrentlyM                                  15 non-null     int64   
 2   Latitude                                    15 non-null     float64 
 3   Longitude                                   15 non-null     float64 
 4   SampleUID                                   15 non-null     object  
 5   Source                                      15 non-null     object  
 6   WesternHem                                  15 non-null     bool    
 7   .geo                                        15 non-null     object  
 8   Category (1=same as YFB lakes/2=different)  13 non-null     float64 
 9   coscatsID                                   2 non-null      float64 
 10  Sbcode 

In [64]:
geocoded.SampleUID

12     BML_00000031
13     BML_00000060
14     BML_00000060
15     BML_00000061
16     BML_00000065
17     BML_00000065
18     BML_00000066
19     BML_00000097
20     BML_00000097
31     BML_00000177
32     BML_00000187
40     BML_00000281
41     BML_00000282
95     BML_00006030
100    BML_00006115
Name: SampleUID, dtype: object

Finally, query which were ultimately labeled as mismatches

In [82]:
comparison = geocoded.merge(gee_output_gdf, left_on="SampleUID", right_on="sampleUID")
comparison.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 38 columns):
 #   Column                                      Non-Null Count  Dtype   
---  ------                                      --------------  -----   
 0   system:index                                20 non-null     object  
 1   CurrentlyM                                  20 non-null     int64   
 2   Latitude                                    20 non-null     float64 
 3   Longitude                                   20 non-null     float64 
 4   SampleUID                                   20 non-null     object  
 5   Source                                      20 non-null     object  
 6   WesternHem                                  20 non-null     bool    
 7   .geo                                        20 non-null     object  
 8   Category (1=same as YFB lakes/2=different)  18 non-null     float64 
 9   coscatsID                                   2 non-null      float64 
 10  Sbco

Aha, four were geocoded

In [84]:
comparison.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 38 columns):
 #   Column                                      Non-Null Count  Dtype   
---  ------                                      --------------  -----   
 0   system:index                                20 non-null     object  
 1   CurrentlyM                                  20 non-null     int64   
 2   Latitude                                    20 non-null     float64 
 3   Longitude                                   20 non-null     float64 
 4   SampleUID                                   20 non-null     object  
 5   Source                                      20 non-null     object  
 6   WesternHem                                  20 non-null     bool    
 7   .geo                                        20 non-null     object  
 8   Category (1=same as YFB lakes/2=different)  18 non-null     float64 
 9   coscatsID                                   2 non-null      float64 
 10  Sbco

In [93]:
comparison["geometry"] = gpd.points_from_xy(comparison["geocode_lat"], comparison["geocode_lon"])

In [94]:
comparison.query("savetype == 'Mismatch'")[
    [
        "Latitude",
        "Longitude",
        "geocode_lat",
        "geocode_lon",
        "SampleUID",
        "pld_match",
        "lake name provided",
        "geocode_full_name",
        "geocoder",
        "geocode_geom",
        "geometry",
    ]
]

,Latitude,Longitude,geocode_lat,geocode_lon,SampleUID,pld_match,lake name provided,geocode_full_name,geocoder,geocode_geom,geometry
16,54.85012,-67.93330,54.850543,-67.870029,BML_00000281,-99999.31415,Lac Chaigneau,"Lac Chaigneau, Caniapiscau, Caniapiscau (MRC),...",nominatim,False,POINT (54.851 -67.87)
17,54.85815,-66.80572,54.853976,-66.918452,BML_00000282,-99999.31415,Lac Ridge,"Lac Ridge, Lac-Vacher, Caniapiscau (MRC), Côte...",nominatim,True,POINT (54.854 -66.918)
18,69.00000,-149.00000,68.984126,-149.296026,BML_00006030,-99999.31415,Toolik,"Toolik River, Alaska, USA",google,False,POINT (68.984 -149.296)
19,71.18000,-156.39000,71.308473,-156.652528,BML_00006115,-99999.31415,North Meadow Lake,"North Meadow Lake, Utqiagvik, AK 99723, USA",google,False,POINT (71.308 -156.653)


In [5]:
# First, set QA pass columns to not geocoded
geocode_fail_idx = geocode_gdf.query("QA == 0").index
geocode_gdf.loc[geocode_fail_idx, ["geocoder", "geocode_full_name", "geocode_lat", "geocode_lon"]] = (
    np.nan
)
geocode_gdf.loc[geocode_fail_idx, ["geocode_geom"]] = False

In [6]:
# How many matched features had no lat/lon? Actually, all have lat/lon
count_bogard_suppl_with_coords = geocode_gdf[
    geocode_gdf["lat (decimal)"].notnull() & geocode_gdf["long (decimal)"].notnull()
].shape[0]

count_geocode_no_coords = geocode_gdf[(geocode_gdf['geocoder'].notnull()) & (geocode_gdf['lat (decimal)'].isnull())].shape[0]
print(
    f"Number of bogard_suppl features with lat/lon: {count_bogard_suppl_with_coords} / {len(geocode_gdf)}"
)
print(
    f"Number of geocoded features with no lat/lon: {count_geocode_no_coords} / {len(geocode_gdf)}"
)

Number of bogard_suppl features with lat/lon: 922 / 922
Number of geocoded features with no lat/lon: 0 / 922
